# V1-22: 기간 Resolver 가용성 정책 비교

## 목적

21번에서 LLM은 `최근 3년`을 `recent_years=3`으로만 구조화했습니다. 이 notebook은 그 요청을 실제 OpenDART 사업보고서 재무 데이터의 **사용 가능한 사업연도**로 바꾸는 결정론적 Period Resolver를 비교합니다.

- **A. 달력 기준**: 기준연도부터 N개 연도를 단순 선택한다. 실제 사업보고서·재무 데이터 존재 여부는 선택 뒤에야 확인한다.
- **B. 가용성·공통연도 기준**: CFS 우선/OFS 전체 대체 규칙으로 실제 재무 행이 있는 연도만 찾고, 다기업 비교는 공통 연도만 선택한다.

이번에는 LLM을 다시 호출하지 않습니다. 21번에서 조건부 채택한 Prompt B가 남긴 기업·기간 구조를 받는 **후속 코드 경로**만 확인합니다. 실제 OpenDART API는 호출하지만 최종 LangChain Tool, 계산, Agent workflow, API·대시보드는 구현하지 않습니다.

> 사용 근거: OpenDART 단일회사 전체 재무제표 API의 사업연도·사업보고서·CFS/OFS 응답 계약은 `docs/v1/REFERENCES.md`에 정리된 공식 개발가이드를 따른다. 04·20번에서 확인한 CFS 우선/OFS 전체 대체와 IS 우선/CIS 대체 원칙은 이 notebook에서 새 후보로 비교하지 않고 고정 입력 정책으로 사용한다.


In [ ]:
from datetime import date, datetime, timezone
import json
import os
from pathlib import Path
import sys
import time

from IPython.display import display

import pandas as pd


In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

if not os.environ.get('OPENDART_API_KEY'):
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

sys.path.insert(0, str(project_root / 'src'))
from dart.client import fetch_financial_rows, find_corp_info

API_KEY = os.environ['OPENDART_API_KEY']
REPORT_CODE = '11011'
# 실행일과 무관하게 비교를 재현하기 위한 평가 기준연도다. 2026년 사업보고서는 아직 존재하지 않아야 한다.
REFERENCE_YEAR = 2026
MAX_LOOKBACK_YEARS = 8
RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(),
    'reference_year': REFERENCE_YEAR,
    'max_lookback_years': MAX_LOOKBACK_YEARS,
    'report_code': REPORT_CODE,
    'cfs_ofs_policy': 'CFS first; OFS only after CFS status 013',
}
print(f'Project root: {project_root}')
print(f'OpenDART API key loaded: {bool(API_KEY)} (값은 출력하지 않음)')
RUN_CONFIG


In [ ]:
# 정답은 기간 정책 계약으로만 적는다. 실제 가용 연도는 API 응답에서 확인하며, 모델 Prompt에는 사용하지 않는다.
EVAL_CASES = [
    {
        'id': 'P01', 'type': 'recent_single', 'companies': ['삼성전자'], 'recent_years': 3, 'explicit_years': [],
        'expected_status': 'OK', 'description': '단일 기업 최근 3개 사업연도',
    },
    {
        'id': 'P02', 'type': 'recent_multi', 'companies': ['삼성전자', 'SK하이닉스'], 'recent_years': 3, 'explicit_years': [],
        'expected_status': 'OK', 'description': '다기업 최근 3개 공통 사업연도',
    },
    {
        'id': 'P03', 'type': 'recent_multi', 'companies': ['삼성전자', '에이피알'], 'recent_years': 3, 'explicit_years': [],
        'expected_status': 'OK_OR_PARTIAL', 'description': '상장 시점·공시 이력이 다른 기업과의 최근 공통연도',
    },
    {
        'id': 'P04', 'type': 'explicit', 'companies': ['리노공업'], 'recent_years': None, 'explicit_years': [2024],
        'expected_status': 'OK', 'description': 'CFS 부재 시 OFS 전체 대체가 확인된 기업의 명시 연도',
    },
    {
        'id': 'P05', 'type': 'explicit', 'companies': ['삼성전자'], 'recent_years': None, 'explicit_years': [2026],
        'expected_status': 'LIMITATION', 'description': '아직 사업보고서가 존재하지 않는 명시 연도',
    },
]

pd.DataFrame(EVAL_CASES)[['id', 'description', 'companies', 'recent_years', 'explicit_years', 'expected_status']]


In [ ]:
company_cache, availability_cache, api_log = {}, {}, []


def get_company(name):
    if name not in company_cache:
        company_cache[name] = find_corp_info(name, API_KEY)
    return company_cache[name]


def fetch_available_statement(name, year):
    """CFS 성공 시 CFS를, OpenDART 013일 때만 OFS를 반환한다. 다른 오류는 가용성으로 숨기지 않는다."""
    key = (name, year)
    if key in availability_cache:
        return availability_cache[key]

    started_at = time.perf_counter()
    try:
        corp = get_company(name)
        rows = fetch_financial_rows(corp['corp_code'], year, API_KEY, REPORT_CODE, 'CFS')
        status = 'AVAILABLE' if rows else 'EMPTY_RESPONSE'
        result = {'company': name, 'corp_code': corp['corp_code'], 'year': year, 'status': status, 'fs_div': 'CFS' if rows else None, 'row_count': len(rows), 'error': None if rows else 'OpenDART status 000 but list is empty'}
    except RuntimeError as cfs_error:
        if '[013]' not in str(cfs_error):
            result = {'company': name, 'corp_code': None, 'year': year, 'status': 'ERROR', 'fs_div': None, 'row_count': 0, 'error': str(cfs_error)}
        else:
            try:
                corp = get_company(name)
                rows = fetch_financial_rows(corp['corp_code'], year, API_KEY, REPORT_CODE, 'OFS')
                status = 'AVAILABLE' if rows else 'EMPTY_RESPONSE'
                result = {'company': name, 'corp_code': corp['corp_code'], 'year': year, 'status': status, 'fs_div': 'OFS' if rows else None, 'row_count': len(rows), 'error': None if rows else 'OpenDART status 000 but list is empty'}
            except RuntimeError as ofs_error:
                status = 'UNAVAILABLE' if '[013]' in str(ofs_error) else 'ERROR'
                result = {'company': name, 'corp_code': corp.get('corp_code'), 'year': year, 'status': status, 'fs_div': None, 'row_count': 0, 'error': str(ofs_error)}
    except Exception as error:
        result = {'company': name, 'corp_code': None, 'year': year, 'status': 'ERROR', 'fs_div': None, 'row_count': 0, 'error': f'{type(error).__name__}: {error}'}

    result['latency_ms'] = (time.perf_counter() - started_at) * 1_000
    availability_cache[key] = result
    api_log.append(result)
    return result


def calendar_baseline(case):
    if case['explicit_years']:
        years = sorted(case['explicit_years'], reverse=True)
    else:
        years = list(range(REFERENCE_YEAR - 1, REFERENCE_YEAR - 1 - case['recent_years'], -1))
    return {'years': years, 'policy': 'calendar_baseline'}


def availability_aware_resolver(case):
    if case['explicit_years']:
        checks = [fetch_available_statement(company, year) for company in case['companies'] for year in case['explicit_years']]
        unavailable = [check for check in checks if check['status'] != 'AVAILABLE']
        scan_errors = [check for check in checks if check['status'] == 'ERROR']
        status = 'ERROR' if scan_errors else ('OK' if not unavailable else 'LIMITATION')
        years = sorted(case['explicit_years'], reverse=True) if status == 'OK' else []
        return {'years': years, 'policy': 'availability_aware', 'status': status, 'checks': checks, 'scan_errors': scan_errors}

    requested = case['recent_years']
    candidate_years = list(range(REFERENCE_YEAR - 1, REFERENCE_YEAR - MAX_LOOKBACK_YEARS - 1, -1))
    availability_by_company, scan_checks, common_years = {}, [], []
    for company in case['companies']:
        availability_by_company[company] = {}

    for year in candidate_years:
        year_checks = []
        for company in case['companies']:
            check = fetch_available_statement(company, year)
            availability_by_company[company][year] = check
            year_checks.append(check)
            scan_checks.append(check)
        scan_errors = [check for check in year_checks if check['status'] == 'ERROR']
        if scan_errors:
            return {'years': [], 'policy': 'availability_aware', 'status': 'ERROR', 'checks': scan_checks, 'scan_errors': scan_errors, 'available_common_years': common_years}
        if all(check['status'] == 'AVAILABLE' for check in year_checks):
            common_years.append(year)
            if len(common_years) == requested:
                break
    selected = common_years[:requested]
    status = 'OK' if len(selected) == requested else 'PARTIAL'
    checks = [availability_by_company[company][year] for company in case['companies'] for year in selected]
    return {'years': selected, 'policy': 'availability_aware', 'status': status, 'checks': checks, 'scan_errors': [], 'available_common_years': common_years}


In [ ]:
# 두 후보는 동일한 OpenDART 응답 cache를 공유한다. A의 선택 연도도 실제로 존재하는지 별도로 확인한다.
run_rows = []
for case in EVAL_CASES:
    print(f'[{case["id"]}] {case["description"]}')
    candidates = {'A. 달력 기준': calendar_baseline(case), 'B. 가용성·공통연도 기준': availability_aware_resolver(case)}
    for candidate_name, result in candidates.items():
        selected_checks = [fetch_available_statement(company, year) for company in case['companies'] for year in result['years']]
        available_count = sum(check['status'] == 'AVAILABLE' for check in selected_checks)
        selected_count = len(case['companies']) * len(result['years'])
        selection_valid = selected_count > 0 and available_count == selected_count
        requested_count_satisfied = (
            len(result['years']) == case['recent_years']
            if case['recent_years'] is not None else None
        )
        expected_statuses = set(case['expected_status'].split('_OR_'))
        status = result.get('status') or ('OK' if selection_valid else 'PARTIAL')
        status_contract_match = status in expected_statuses
        limitation_correct = (case['expected_status'] == 'LIMITATION' and status == 'LIMITATION') or (case['expected_status'] != 'LIMITATION')
        run_rows.append({
            'candidate': candidate_name, 'case_id': case['id'], 'companies': case['companies'],
            'request': f"explicit={case['explicit_years']} / recent={case['recent_years']}",
            'selected_years': result['years'], 'resolver_status': status,
            'selected_company_years': selected_count, 'available_company_years': available_count,
            'available_year_valid': selection_valid, 'requested_count_satisfied': requested_count_satisfied,
            'status_contract_match': status_contract_match, 'limitation_correct': limitation_correct,
            'statement_bases': sorted({check['fs_div'] for check in selected_checks if check['fs_div']}),
            'errors': [check['error'] for check in selected_checks if check['status'] == 'ERROR'] + [check['error'] for check in result.get('scan_errors', [])],
        })

results = pd.DataFrame(run_rows)
display(results[['candidate', 'case_id', 'companies', 'request', 'selected_years', 'resolver_status', 'available_company_years', 'selected_company_years', 'available_year_valid', 'requested_count_satisfied', 'status_contract_match', 'statement_bases', 'errors']])

api_frame = pd.DataFrame(api_log)
display(api_frame[['company', 'year', 'status', 'fs_div', 'row_count', 'latency_ms', 'error']].sort_values(['company', 'year'], ascending=[True, False]))


In [ ]:
summary_rows = []
for candidate, group in results.groupby('candidate', sort=False):
    multi = group[group['companies'].map(len) > 1]
    summary_rows.append({
        '후보': candidate,
        'Available-year Validity Rate': (
            group.loc[group['selected_company_years'] > 0, 'available_company_years'].sum()
            / group.loc[group['selected_company_years'] > 0, 'selected_company_years'].sum()
            if (group['selected_company_years'] > 0).any() else None
        ),
        'Requested-year Count Satisfaction': group['requested_count_satisfied'].dropna().mean(),
        'Common-Year Validity': multi.loc[multi['selected_company_years'] > 0, 'available_year_valid'].mean() if (multi['selected_company_years'] > 0).any() else None,
        'Resolver Status Contract Match': group['status_contract_match'].mean(),
        'Limitation Accuracy': group.loc[group['case_id'] == 'P05', 'limitation_correct'].mean(),
        'API 오류 사례 수': int(group['errors'].map(len).sum()),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

failures = results[(results['selected_company_years'] > 0) & (~results['available_year_valid']) | (results['requested_count_satisfied'] == False) | (~results['status_contract_match']) | (results['errors'].map(len) > 0)]
if failures.empty:
    print('선택된 모든 기업·연도 조합에서 재무제표 가용성이 확인됐습니다.')
else:
    print('아래는 단순 달력 선택 또는 가용성 탐색이 부족한 사례입니다. selected_years와 API 상태를 함께 확인하세요.')
    display(failures)

print('해석 원칙: B가 실제 가용성·공통연도 지표에서 명확히 우세하고, 탐색 중 API 오류를 숨기지 않을 때만 B를 Period Resolver 후보로 둡니다. 표본에서 차이가 없으면 자동 채택하지 않고, 데이터 부재·신규 상장 등 더 적절한 사례를 먼저 검토합니다.')


## 실행 후 확인할 점

- A가 선택한 연도 중 실제 OpenDART 재무 행이 없는 연도가 있는지
- B가 단일 기업에서는 요청한 최근 N개 연도를 모두 찾는지, 다기업에서는 같은 연도만 선택하는지
- P04에서 리노공업이 OFS 기준으로 남는지
- P05의 2026년 명시 요청을 숫자로 대체하지 않고 `LIMITATION`으로 남기는지

실제 A/B 비교 결과가 나온 뒤에만 HTML 보고서와 Period Resolver 후보의 채택·미채택을 기록합니다.
